In [4]:
import sys
import os
from pathlib import Path
import torch
import copy
import numpy as np
from dotenv import load_dotenv

sys.path.append(os.path.join('..', '..'))
env_file = os.path.join('..', '..', '.env')
load_dotenv(env_file)

# Кеш----------------------------------------------------------------------------
doclaynet_cash_pdf_path = os.environ['DOCLAYNET_CASH_PDF_PATH']
publaynet_cash_pdf_path = os.environ['PUBLAYNET_CASH_PDF_PATH']
# -------------------------------------------------------------------------------

# Тренеровка --------------------------------------------------------------------
doclaynet_train_pdf_path = os.environ['DOCLAYNET_TRAIN_PDF_PATH']
doclaynet_train_coco_path = os.environ['DOCLAYNET_TRAIN_COCO_PATH']

publaynet_train_pdf_path = os.environ['PUBLAYNET_TRAIN_PDF_PATH']
publaynet_train_coco_path = os.environ['PUBLAYNET_TRAIN_COCO_PATH']
# -------------------------------------------------------------------------------

# Тестирование ------------------------------------------------------------------
doclaynet_test_pdf_path = os.environ['DOCLAYNET_TEST_PDF_PATH']
doclaynet_test_coco_path = os.environ['DOCLAYNET_TEST_COCO_PATH']

publaynet_test_pdf_path = os.environ['PUBLAYNET_TEST_PDF_PATH']
publaynet_test_coco_path = os.environ['PUBLAYNET_TEST_COCO_PATH']
# -------------------------------------------------------------------------------


from rows2regionsGLAM.utils.pdf_manager import PDFManager
from rows2regionsGLAM.utils.loger import Loger
from rows2regionsGLAM.utils.row_manager import RowManager
from rows2regionsGLAM.utils.ploter import Ploter
from rows2regionsGLAM.utils.trainer import Trainer
from rows2regionsGLAM.utils.tester import Tester
from rows2regionsGLAM.utils.cacher import Cacher
from rows2regionsGLAM.utils.coco_manager import COCOManager
from rows2regionsGLAM.utils.imbalance import calculate_imbalance
from rows2regionsGLAM.utils.tester import collect_maps, print_map_table
from rows2regionsGLAM.tokenizers import RowGLAMTokenizer
from rows2regionsGLAM.models import get_loss, get_model, get_tmp_params


from rows2regionsGLAM.converters import Rows2Regions
from rows2regionsGLAM.datasetloaders.base_line_dataset import GLAMDataset
from pager.page_model.sub_models.dtype import ImageSegment
from pager.page_model.sub_models import RegionModel, RowsModel

import datetime 
loger = Loger(f'log_{datetime.datetime.now()}.txt')
pdf_manager = PDFManager(conf={"loger": loger, "pdf_reader": "PDFMiner"})
row_manager = RowManager(conf={"loger": loger, "add_image": True})

pub_coco_manager = COCOManager(conf={"loger": loger, "coco_path": publaynet_coco_path})
doc_coco_manager = COCOManager(conf={"loger": loger, "coco_path": doclaynet_coco_path})

ploter = Ploter(conf={"loger": loger})
tokenizer = RowGLAMTokenizer()
pdf2torch_dict = Cacher({
        "loger": loger,
        "pdf_manager": pdf_manager,
        "row_manager": row_manager,
        "tokenizer": tokenizer
})

json_true_regions, PUB_CLASSES = pub_coco_manager.get_regions_from_json()
PUB_CLASSES[3] = "text"


json_true_regions, DOC_CLASSES = doc_coco_manager.get_regions_from_json()


from utils.experimenter import Experimenter
PATH_EXP = 'test_path'
exp = Experimenter(name='test', result_save_path=PATH_EXP)

In [ ]:
dict_params = {
    train_ds+"_"+test_ds: {
        "model_param": {"train_ds": train_ds, 'test_ds': test_ds},
        "dataset_param": {"train_ds": train_ds, 'test_ds': test_ds},
        "train_param": {"train_ds": train_ds, 'test_ds': test_ds},
        "test_param": {"train_ds": train_ds, 'test_ds': test_ds}
    }
    for train_ds in ['train_pub', 'train_doc'] for test_ds in ['test_pub', 'test_doc']
}

def fun_get_dataset_with_param(param):
    train_ds = param["train_ds"]
    test_ds = param["test_ds"]
    
    if train_ds=='train_pub':
        dataset = GLAMDataset({
            "loger": loger,
            "pdf_dir": publaynet_train_pdf_path,
            "coco_file": publaynet_coco_path,
            "count_class": len(PUB_CLASSES),
            "name_dataset": "publaynet",
            "default_index": 0,
            "cache_dir": publaynet_train_cash_pdf_path,
            "pdf2torch_dict": pdf2torch_dict
        })
    elif train_ds=='train_doc':
        dataset = GLAMDataset({
            "loger": loger,
            "pdf_dir": doclaynet_train_pdf_path,
            "coco_file": doclaynet_coco_path,
            "count_class": len(DOC_CLASSES),
            "name_dataset": "doclaynet",
            "default_index": 0,
            "cache_dir": doclaynet_train_cash_pdf_path,
            "pdf2torch_dict": pdf2torch_dict
        })
    else:
        raise Exception('неверная конфигурация')


    # Создание Cache
    N = len(dataset)
    for i, d in enumerate(dataset):
        print(f"{(i+1)/N*100:4.2f} %", end='\r')

    if train_ds=='train_pub' and test_ds=='test_pub':
        test_dataset = GLAMDataset(
            {
            "loger": loger,
            "pdf_dir": publaynet_test_pdf_path,
            "coco_file": publaynet_test_coco_path,
            "count_class": len(PUB_CLASSES),
            "name_dataset": "publaynet",
            "default_index": 0,
            "cache_dir": publaynet_test_cash_pdf_path,
            "pdf2torch_dict": pdf2torch_dict
            }
        )
    elif train_ds=='train_pub' and test_ds=='test_doc':
        test_dataset = GLAMDataset(
            {
            "loger": loger,
            "pdf_dir": doclaynet_test_pdf_path,
            "coco_file": doclaynet_test_coco_path_for_pub,
            "count_class": len(DOC_CLASSES),
            "name_dataset": "doclaynet",
            "default_index": 0,
            "cache_dir": doclaynet_test_cash_pdf_path_for_pub,
            "pdf2torch_dict": pdf2torch_dict
            }
        )
    elif train_ds=='train_doc' and test_ds=='test_pub':
        
    elif train_ds=='train_doc' and test_ds=='test_doc':
        
    else:
        raise Exception('неверная конфигурация')
    # Создание Cache
    N = len(test_dataset)
    for i, d in enumerate(test_dataset):
        print(f"{(i+1)/N*100:4.2f} %", end='\r')

    
    return {
        "train": dataset,
        "test": test_dataset
    }
        

def fun_get_model_with_param(param):
    type_model=param["type"]
    exist_font=param["exist_font"]

    
    model_name = str(Path(PATH_EXP, f'row2region_GLAM_{type_model}_{exist_font}'))
    
    if type_model == "base":
        model_params = get_tmp_params(type_model)
    elif type_model == "custom":
        model_params  = {
            "edge_featch": 4,
            "learning_rate": 0.001,
            "model_type" : 2,
            "concat_gcn" : True,
            "mlp_pred":[
                 {"in" : -1, "batch_norm" : False, "activation" : "gelu", "out" : 256},
                 {"in" : -1, "batch_norm" : False, "activation" : "gelu", "out" : 128}],
            "gcn_node":[
                 {"linear_in" : -1, "linear_out" : 256, "batch_norm" : True,  "activation" : "gelu", "aggregation" : "tag", "K" : 3,  "size":128},
                 {"linear_in" : -1, "linear_out" : 256, "batch_norm" : False, "activation" : "gelu", "aggregation" : "tag", "K" : 3,  "size":64}],
            "mlp_node_class":[
                 {"in" : -1, "batch_norm" : False, "activation" : "gelu", "out" : 256},
                 {"in" : -1, "batch_norm" : False, "activation" : "gelu", "out" : 128},
                 {"in" : -1, "batch_norm" : False, "activation" : "softmax", "out" : 6}],
            "mlp_node_pred":[
                 {"in" : -1, "batch_norm" : False, "activation" : "gelu", "out" : 256},
                 {"in" : -1, "batch_norm" : False, "activation" : "gelu", "out" : 256},
                 {"in" : -1, "batch_norm" : False, "activation" : "gelu", "out" : 128}],
            "gcn_node_post":[
                 {"linear_in" : -1, "linear_out" : 256, "batch_norm" : True,  "activation" : "gelu", "aggregation" : "tag", "K" : 3,  "size":128},
                 {"linear_in" : -1, "linear_out" : 256, "batch_norm" : False, "activation" : "gelu", "aggregation" : "tag", "K" : 3,  "size":128}],
            "mlp_node_edge":[
                 {"in" : -1, "batch_norm" : False, "activation" : "gelu", "out" : 256},
                 {"in" : -1, "batch_norm" : False, "activation" : "gelu", "out" : 128},
                 {"in" : -1, "batch_norm" : False, "activation" : None,   "out" : 128}],
            "mlp_edge_class":[
                 {"in" : -1, "batch_norm" : False, "activation" : "gelu", "out" : 256},
                 {"in" : -1, "batch_norm" : False, "activation" : "gelu", "out" : 64},
                 {"in" : -1, "batch_norm" : False, "activation" : None,   "out" : 1}],
            "seg_k": 0.5,
            "loss_params": {
                "edge_coef": 0.8,
                "node_coef": 0.2,
            },
            "sigmoidEdge": False,
            "NodeClasses": len(CLASSES)
        }
    else:
        raise Exception('неверная конфигурация')


    if exist_font == 'font':
        model_params["node_featch"] = 18
    elif exist_font == 'no_font':
        model_params["node_featch"] = 15
    else:
        raise Exception('неверная конфигурация')
    
    model_params["NodeClasses"] = len(CLASSES)
    model_params["epochs"] = 10
    model_params["batch_size"] = 64

    return {
         'model_name': model_name,
         'model_params': model_params
    }

def fun_train_model_with_param(model, dataset, param):

    model_name = model['model_name']
    model_params = model['model_params']
    
    dataset = dataset['train']
    type_model=param["type"]
    
    publaynet_imbalance, edge_imbalance = calculate_imbalance(dataset)
    model_params['loss_params']['publaynet_imbalance'] = publaynet_imbalance
    model_params['loss_params']['edge_imbalance'] = edge_imbalance

    if not Path(model_name).exists():
        trainer_pub = Trainer(conf={"loger": loger, "params": model_params, "model_name": model_name})
        trainer_pub.start_train(5, dataset, type_model=type_model)
    
def fun_test_model_with_param(model, dataset, param):
    test_dataset = dataset['test']
    type_model=param["type"]
    exist_font =  param['exist_font']
    model_name = model['model_name']
    model_params = model['model_params']

    
    if exist_font == 'font':
        tokenizer = font_tokenizer
    elif exist_font == 'no_font':
        tokenizer = no_font_tokenizer
    else:
        raise Exception('неверная конфигурация')
    
    model_params['sigmoidEdge'] = True
    model = get_model(type_model, model_params)
    model.load_state_dict(torch.load(model_name, weights_only=True))
    
    rows_model = RowsModel()
    region_model = RegionModel()
    rows2regions = Rows2Regions({
        'model':model, 
        'tokenizer': tokenizer,
        'is_merge_extract': True,
        'classes': CLASSES
    })
    tester = Tester(conf={
        "loger": loger, 
        "pdf_manager": pdf_manager, 
        "row_manager": row_manager, 
        "rows_model": rows_model, 
        "region_model": region_model, 
        "rows2regions": rows2regions})

    metrics = tester.calculate_target_and_preds(
        test_dataset, 
        name_dataset, 
        name_test_dataset, 
        dataset_path, 
        test_path
    )

    mAP, grid = tester.get_results(metrics)
    return {"map": mAP.split(':')[-1], "grid": grid }

def fun_result_to_row(train_result, test_result):
    mAP = test_result['map']
    grid = test_result['grid']
    return {
        "mAP@IoU[0.50:0.95]": float(mAP),
        "F1@IoU_row[0.50]": grid['threshold_05']['f1_row'],
        "F1@IoU_row[0.95]": grid['threshold_95']['f1_row']
    }


exp.experiment(
    fun_get_model_with_param, 
    fun_get_dataset_with_param,
    fun_train_model_with_param,
    fun_test_model_with_param,
    fun_result_to_row,
    dict_params
)